In [0]:
%sql
-- Creating a catalog and schema 
DROP table if exists cdc_catalog.cdc_schema.customer_source_table;
DROP table if exists cdc_catalog.cdc_schema.users_current;
DROP table if exists cdc_catalog.cdc_schema.users_history;
DROP table if exists cdc_catalog.cdc_schema.customers_cdf;
Create catalog if not exists cdc_catalog;
create schema if not exists cdc_catalog.cdc_schema;
create volume if not exists cdc_catalog.cdc_schema.cdc_customer_vol_ckpoint ;
GRANT ALL PRIVILEGES ON catalog  cdc_catalog TO `account users`;
GRANT ALL PRIVILEGES ON SCHEMA  cdc_catalog.cdc_schema TO `account users`;
use cdc_catalog.cdc_schema;

In [0]:
%sql
use cdc_catalog.cdc_schema;

In [0]:
print('The below path will drop the checkout path...by default it will be disabled')
chkpnt_path = '/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint'
print(f"The checkpoint path :-------- {chkpnt_path}")
#dbutils.fs.rm(f"{chkpnt_path}",recurse=True)

The below path will drop the checkout path...by default it will be disabled
The checkpoint path :-------- /Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint


In [0]:
%sql
select current_schema(),current_catalog()

current_schema(),current_catalog()
cdc_schema,cdc_catalog


In [0]:
%sql
SELECT 
    grantee, 
    privilege_type, 
    'CATALOG' AS object_type, 
    catalog_name AS object_name
FROM information_schema.catalog_privileges
WHERE catalog_name = 'cdc_catalog'

UNION ALL

SELECT 
    grantee, 
    privilege_type, 
    'SCHEMA' AS object_type, 
    schema_name AS object_name
FROM information_schema.schema_privileges
WHERE catalog_name = 'cdc_catalog' 
  AND schema_name = 'cdc_schema';

grantee,privilege_type,object_type,object_name
account users,ALL_PRIVILEGES,CATALOG,cdc_catalog
account users,ALL_PRIVILEGES,SCHEMA,cdc_schema
account users,ALL_PRIVILEGES,SCHEMA,cdc_schema


In [0]:
%sql
Create table customer_source_table(
  userId  INT,name STRING , city STRING)
  USING DELTA 
  TBLPROPERTIES (delta.enableChangeDataFeed = true,
                 delta.deletedFileRetentionDuration = 'interval 1 days');

In [0]:
def calling_customer_cdf():
    spark.\
        readStream.\
            option("readChangeFeed", "true").\
                option("startingVersion", 0).\
                    table("customer_source_table").\
                        writeStream.\
                            option("checkpointLocation", "/Volumes/cdc_catalog/cdc_schema/cdc_customer_vol_ckpoint").\
                                trigger (availableNow=True).\
                                    table("customers_cdf")

In [0]:
%sql
-----------Inserting data in to customer_source_table
INSERT INTO  customer_source_table
SELECT
  col1 AS userId,
  col2 AS name,
  col3 AS city
FROM (
  VALUES
  -- Initial load.
  (101, "Raul",     "Oaxaca"),
  (102, "Isabel",   "Monterrey"),
  (103, "Mercedes", "Tijuana"),
  (104, "Lily",     "Cancun")
);

num_affected_rows,num_inserted_rows
4,4


In [0]:
# running the streaming
calling_customer_cdf()

In [0]:
%sql
---Checking data ---
select * from customers_cdf

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,null
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,null
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,null
104,Lily,Cancun,2026-08-04T09:19:46.000Z,null


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city
101,Raul,Oaxaca
102,Isabel,Monterrey
103,Mercedes,Tijuana
104,Lily,Cancun


In [0]:
%sql
INSERT INTO  cdc_catalog.cdc_schema.customer_source_table VALUES (105, "Jacob","Florida")

num_affected_rows,num_inserted_rows
1,1


In [0]:
calling_customer_cdf()

In [0]:
%sql
select * from cdc_catalog.cdc_schema.customers_cdf;

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z
105,Jacob,Florida,insert,2,2026-08-04T17:11:40.000Z


# RUN The pipeline

In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,null
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,null
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,null
104,Lily,Cancun,2026-08-04T09:19:46.000Z,null
105,Jacob,Florida,2026-08-04T17:11:40.000Z,null


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city
101,Raul,Oaxaca
102,Isabel,Monterrey
103,Mercedes,Tijuana
104,Lily,Cancun
105,Jacob,Florida


# Update the records


In [0]:
%sql
update cdc_catalog.cdc_schema.customer_source_table
set city ='Columbus' 
where userId =102

num_affected_rows
1


In [0]:
calling_customer_cdf()

In [0]:
%sql
select * from cdc_catalog.cdc_schema.customers_cdf;

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,update_preimage,3,2026-08-04T17:23:27.000Z
102,Isabel,Columbus,update_postimage,3,2026-08-04T17:23:27.000Z
105,Jacob,Florida,insert,2,2026-08-04T17:11:40.000Z


# RUN The pipeline again

In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
102,Isabel,Columbus,2026-08-04T17:23:27.000Z,null
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,2026-08-04T17:23:27.000Z
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,null
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,null
104,Lily,Cancun,2026-08-04T09:19:46.000Z,null
105,Jacob,Florida,2026-08-04T17:11:40.000Z,null


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city
101,Raul,Oaxaca
103,Mercedes,Tijuana
104,Lily,Cancun
105,Jacob,Florida
102,Isabel,Columbus


# Delete the records

In [0]:
%sql
delete from cdc_catalog.cdc_schema.customer_source_table
where userId =104

num_affected_rows
1


In [0]:
calling_customer_cdf()

In [0]:
%sql
select * from cdc_catalog.cdc_schema.customers_cdf;

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,update_preimage,3,2026-08-04T17:23:27.000Z
102,Isabel,Columbus,update_postimage,3,2026-08-04T17:23:27.000Z
103,Mercedes,Tijuana,delete,5,2026-08-04T17:35:11.000Z
105,Jacob,Florida,insert,2,2026-08-04T17:11:40.000Z
104,Lily,Cancun,delete,7,2026-08-04T17:49:04.000Z


# Run the pipeline

In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city
101,Raul,Oaxaca
105,Jacob,Florida
102,Isabel,Columbus
103,Mercedes,Tijuana


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
102,Isabel,Columbus,2026-08-04T17:23:27.000Z,null
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,2026-08-04T17:23:27.000Z
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,null
104,Lily,Cancun,2026-08-04T09:19:46.000Z,2026-08-04T17:49:04.000Z
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,null
105,Jacob,Florida,2026-08-04T17:11:40.000Z,null


# Truncate the records

In [0]:
%sql
Truncate table cdc_catalog.cdc_schema.customer_source_table;

In [0]:
calling_customer_cdf()

In [0]:
%sql
select * from cdc_catalog.cdc_schema.customers_cdf;

userId,name,city,_change_type,_commit_version,_commit_timestamp
101,Raul,Oaxaca,insert,1,2026-08-04T09:19:46.000Z
102,Isabel,Monterrey,insert,1,2026-08-04T09:19:46.000Z
103,Mercedes,Tijuana,insert,1,2026-08-04T09:19:46.000Z
104,Lily,Cancun,insert,1,2026-08-04T09:19:46.000Z
101,Raul,Oaxaca,delete,9,2026-08-04T18:06:14.000Z
102,Isabel,Columbus,delete,9,2026-08-04T18:06:14.000Z
105,Jacob,Florida,delete,9,2026-08-04T18:06:14.000Z
102,Isabel,Monterrey,update_preimage,3,2026-08-04T17:23:27.000Z
102,Isabel,Columbus,update_postimage,3,2026-08-04T17:23:27.000Z
103,Mercedes,Tijuana,delete,5,2026-08-04T17:35:11.000Z


# Run the pipeline

In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_current").display()

userId,name,city


In [0]:
spark.sql("select * from cdc_catalog.cdc_schema.users_history").display()

userId,name,city,__START_AT,__END_AT
101,Raul,Oaxaca,2026-08-04T09:19:46.000Z,2026-08-04T18:06:14.000Z
102,Isabel,Monterrey,2026-08-04T09:19:46.000Z,2026-08-04T17:23:27.000Z
102,Isabel,Columbus,2026-08-04T17:23:27.000Z,2026-08-04T18:06:14.000Z
103,Mercedes,Tijuana,2026-08-04T09:19:46.000Z,2026-08-04T17:35:11.000Z
104,Lily,Cancun,2026-08-04T09:19:46.000Z,2026-08-04T17:49:04.000Z
105,Jacob,Florida,2026-08-04T17:11:40.000Z,2026-08-04T18:06:14.000Z
